In [ ]:

import tensorflow as tf

print("TensorFlow version :", tf.__version__)
print("GPU disponible :", tf.config.list_physical_devices('GPU'))


In [ ]:

# Kaninstalliw kagglehub bach ntelechargiw dataset mn Kaggle
!pip -q install kagglehub

# Kanimportiw kagglehub bach nst3mloh f Python
import kagglehub

In [ ]:
import os

# Kantelechargiw dataset dyal les sports
dataset_path = kagglehub.dataset_download(
    "gpiosenka/sports-classification"
)

# Kanchoufo fin t7ett dataset f Colab
print("Dataset path :", dataset_path)

# Kanchoufo chno kayn dakhel dossier
print("Les fichiers :", os.listdir(dataset_path))


In [ ]:

# Kan9elbo 3la dossier train f dataset
for dossier, sous_dossiers, fichiers in os.walk(dataset_path):

    # Ila l9ina dossier smito train
    if os.path.basename(dossier) == "train":

        # Kan7afdo l'adresse dyalo
        train_path = dossier

        # Kan7ebso l9elb hit l9inah
        break

# Kanchoufo fin kayn dossier train
print("Dossier train :", train_path)

# Kanjibo smiyat les catégories li dakhel train
sports = os.listdir(train_path)

# Kanweriw les catégories
print("Les sports disponibles :", sports)


In [ ]:
# Hado homa les 3 sports li bghina nkhdmo bihom
selected_sports = ["football", "basketball", "tennis"]

# Kanweriwhom bach nt2kdo
print("Les sports sélectionnés :", selected_sports)

# Kan7seb ch7al mn tswira kayna f train f kol sport
for sport in selected_sports:
    sport_path = os.path.join(train_path, sport)   # kanjib l'adresse dyal dossier dyal had sport
    images_count = len(os.listdir(sport_path))     # kan7seb ch7al mn fichier kayn dakhel dossier
    print(f"{sport} : {images_count} images")

In [ ]:
# Kanjibo l'adresse dyal dossier principal
base_path = os.path.dirname(train_path)

# Kan7eddo fin kaynin les images dyal training
train_dir = os.path.join(base_path, "train")

# Kan7eddo fin kaynin les images dyal validation
valid_dir = os.path.join(base_path, "valid")

# Kan7eddo fin kaynin les images dyal test
test_dir = os.path.join(base_path, "test")

# Kanchoufo wach les 3 dossiers kaynin
print("Train :", os.path.isdir(train_dir))
print("Validation :", os.path.isdir(valid_dir))
print("Test :", os.path.isdir(test_dir))

In [ ]:
# Had boucle katdouz 3la Train, Validation w Test
for nom, dossier in [
    ("Train", train_dir),
    ("Validation", valid_dir),
    ("Test", test_dir)
]:

    # Kanweriw smit partie
    print("\n", nom)

    # Kan7esbo les images dyal kol sport
    for sport in selected_sports:

        # Kanjibo dossier dyal had sport
        sport_path = os.path.join(dossier, sport)

        # Kan7esbo ch7al mn image fih
        nb_images = len(os.listdir(sport_path))

        # Kanweriw résultat
        print(sport, ":", nb_images, "images")

In [ ]:

# Kanimportiw TensorFlow bach nkhdmo b les images
import tensorflow as tf

# Kan7eddo taille dyal kol image
image_size = (224, 224)

# Kan7eddo ch7al mn image ytraitew f kol batch
batch_size = 16

# Kan7eddo les sports b nafs tartib f ga3 les datasets
classes = ["basketball", "football", "tennis"]

# Kanchargiw les images dyal training
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",          # Labels kayjiw mn smiyat dossiers
    class_names=classes,        # Ghadi nkhdmo gha b 3 sports
    image_size=image_size,      # Kanredou ga3 tsawer 224x224
    batch_size=batch_size,      # 16 tswira f kol batch
    shuffle=True,               # Kanخلطو tsawer bach training ykoun mzyan
    seed=42                     # Bach nkhdmo b nafs random results
)

# Kanchargiw les images dyal validation
valid_ds = tf.keras.utils.image_dataset_from_directory(
    valid_dir,
    labels="inferred",
    class_names=classes,
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False
)

# Kanchargiw les images dyal test
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    class_names=classes,
    image_size=image_size,
    batch_size=batch_size,
    shuffle=False
)

# Kanweriw les catégories li ghadi yst3melhom modèle
print("Classes :", train_ds.class_names)


In [ ]:

# Kanimportiw MobileNetV2, modèle CNN déjà entraîné
from tensorflow.keras.applications import MobileNetV2

# Kanchargiw modèle avec ses poids déjà appris
base_model = MobileNetV2(
    weights="imagenet",      # Kanst3mlo les poids li deja t3ellem
    include_top=False,       # Kan7iydo classification l9dima
    input_shape=(224, 224, 3) # Taille dyal tsawer dyalna
)

# Kanbloquiw les poids dyal MobileNetV2
# Bach man3awdouch ntrainiw modèle kamel
base_model.trainable = False

# Kanchoufo résumé dyal modèle
base_model.summary()


In [ ]:
# Kanimportiw les couches li ghadi nst3mlo
from tensorflow.keras import layers, Model

# Kan7eddo taille dyal tsawer
inputs = layers.Input(shape=(224, 224, 3))

# Kan7awlo pixels mn 0-255 l -1 7ta 1
x = layers.Rescaling(1./127.5, offset=-1)(inputs)

# Kanst3mlo MobileNetV2 bach yextracti les caractéristiques
x = base_model(x, training=False)

# Kansaghro Feature Maps l vecteur dyal 1280 valeurs
x = layers.GlobalAveragePooling2D()(x)

# Kanzido Dense bach nclassifiw 3 sports
outputs = layers.Dense(3, activation="softmax")(x)

# Kanjme3o ga3 les couches f modèle wa7d
model = Model(inputs, outputs)

# Kanweriw résumé dyal modèle
model.summary()

In [ ]:
# Kanwajdo modèle bach ybda yt3ellem
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),  # Adam kay7essen les poids
    loss="sparse_categorical_crossentropy",                  # Kat7seb l'erreur
    metrics=["accuracy"]                                     # Kat7seb prédictions s7a7
)

print("Model compilé avec succès !")

In [ ]:
# Kantrainiw modèle 3la images dyalna
history = model.fit(
    train_ds,                  # 491 images bach yt3ellem
    validation_data=valid_ds,  # 15 images bach n9isso performance
    epochs=10                  # Ghadi ydouz 10 mrat 3la training dataset
)

In [ ]:

# Kanjarrbo modèle 3la images dyal Test
test_loss, test_accuracy = model.evaluate(test_ds)

# Kanweriw pourcentage dyal prédictions s7i7a
print("Test Accuracy :", test_accuracy * 100, "%")

# Kanweriw ch7al l'erreur dyal modèle
print("Test Loss :", test_loss)


In [ ]:

# Kanimportiw numpy bach nkhdmo b les tableaux
import numpy as np

# Hna ghadi njme3o les vraies réponses
y_true = []

# Hna ghadi njme3o les réponses dyal modèle
y_pred = []

# Kandouzou 3la kol batch f Test
for images, labels in test_ds:

    # Modèle kay3ti probabilité dyal kol sport
    predictions = model.predict(images, verbose=0)

    # Kanakhdo sport li 3ndo akbar probabilité
    predicted_classes = np.argmax(predictions, axis=1)

    # Kanjme3o les vraies classes
    y_true.extend(labels.numpy())

    # Kanjme3o les classes li tوقعhom modèle
    y_pred.extend(predicted_classes)

# Kanweriw les résultats
print("Vraies classes :", y_true)
print("Prédictions :", y_pred)


In [ ]:

# Kanimportiw l'outil li kay7seb Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Kan7esbo Confusion Matrix mn les vraies réponses w prédictions
cm = confusion_matrix(y_true, y_pred)

# Kanwajdo l'affichage m3a smiyat les sports
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

# Kanrsmo Confusion Matrix
disp.plot(cmap="Blues")


In [ ]:

# Kanimportiw json bach nsajlo smiyat les sports
import json

# Kansajlo CNN dyalna f fichier keras
model.save("sports_model.keras")

# Kansajlo smiyat les classes b nafs tartib
with open("labels.json", "w") as fichier:
    json.dump(classes, fichier)

# Kanweriw message bach nt2kdo
print("Modèle w labels tsajlo b succès !")


In [ ]:
# Kanimportiw outil bach ntelechargiw les fichiers
from google.colab import files

# Kantelechargiw modèle CNN
files.download("sports_model.keras")

# Kantelechargiw smiyat les sports
files.download("labels.json")

In [ ]:
# Kanimportiw outil bach nuploadiw tswira mn PC
from google.colab import files

# Kanchoisiw tswira mn PC dyalna
uploaded = files.upload()

# Kanakhdo smit tswira li uploadina
image_path = list(uploaded.keys())[0]

# Kanweriw smit tswira
print("Image :", image_path)

In [ ]:
# Kanimportiw numpy bach n7elllo les résultats
import numpy as np

# Kanchargiw tswira w kanredouha 224x224
img = tf.keras.utils.load_img(
    image_path,
    target_size=(224, 224),
    color_mode="rgb"
)

# Kan7awlo tswira l tableau dyal pixels
img_array = tf.keras.utils.img_to_array(img)

# Kanzido dimension dyal batch
img_array = np.expand_dims(img_array, axis=0)

# Modèle kaytwa9e3 sport li f tswira
prediction = model.predict(img_array)

# Kanakhdo sport li 3ndo akbar probabilité
class_index = np.argmax(prediction[0])

# Kanjibo smit sport
sport = classes[class_index]

# Kanweriw résultat
print("Sport prédit :", sport)
print("Confiance :", round(float(prediction[0][class_index]) * 100, 2), "%")